In [2]:
import pandas as pd
df = pd.read_csv("data/RAW_recipes.csv")
print(df.shape)
df.head()

import ast

df = df.dropna(subset=["name", "ingredients", "steps"])

for col in ["tags", "steps", "ingredients", "nutrition"]:
    df[col] = df[col].apply(ast.literal_eval)

df["name"] = df["name"].str.split().str.join(" ")

nutri_cols = ["calories", "fat_pdv", "sugar_pdv", "sodium_pdv", "protein_pdv", "sat_fat_pdv", "carbs_pdv"]
df[nutri_cols] = pd.DataFrame(df["nutrition"].tolist(), index=df.index)

df = df[(df["calories"] > 0) & (df["calories"] < 3000) & (df["minutes"] < 600)]

print(df.shape)
df[["name", "calories", "protein_pdv", "minutes", "n_ingredients"]].head()


(231637, 12)
(225040, 19)


,name,calories,protein_pdv,minutes,n_ingredients
0,arriba baked winter squash mexican style,51.5,2.0,55,7
1,a bit different breakfast pizza,173.4,22.0,30,6
2,all in the kitchen chili,269.8,39.0,130,13
3,alouette potatoes,368.1,14.0,45,11
4,amish tomato ketchup for canning,352.9,3.0,190,8


In [ ]:
def build_chunk(row):
    steps = " ".join(f"{i+1}. {s}" for i, s in enumerate(row["steps"][:8]))
    return (
        f"Recipe: {row['name'].title()}\n"
        f"Description: {str(row['description'])[:300]}\n"
        f"Ingredients: {', '.join(row['ingredients'])}\n"
        f"Tags: {', '.join(row['tags'][:15])}\n"
        f"Steps: {steps}"
    )

df["chunk"] = df.apply(build_chunk, axis=1)
print(df["chunk"].iloc[0])


Recipe: Arriba Baked Winter Squash Mexican Style
Description: autumn is my favorite time of year to cook! this recipe 
can be prepared either spicy or sweet, your choice!
two of my posted mexican-inspired seasoning mix recipes are offered as suggestions.
Ingredients: winter squash, mexican seasoning, mixed spice, honey, butter, olive oil, salt
Tags: 60-minutes-or-less, time-to-make, course, main-ingredient, cuisine, preparation, occasion, north-american, side-dishes, vegetables, mexican, easy, fall, holiday-event, vegetarian
Steps: 1. make a choice and proceed with recipe 2. depending on size of squash , cut into half or fourths 3. remove seeds 4. for spicy squash , drizzle olive oil or melted butter over each cut squash piece 5. season with mexican seasoning mix ii 6. for sweet squash , drizzle melted honey , butter , grated piloncillo over each cut squash piece 7. season with sweet mexican spice mix 8. bake at 350 degrees , again depending on size , for 40 minutes up to an hour , unt

In [5]:
sample = df.sample(3000, random_state=42)

cols = ["id", "name", "chunk", "steps", "calories", "protein_pdv", "fat_pdv", "sugar_pdv",
        "carbs_pdv", "minutes", "n_ingredients", "ingredients", "tags"]
sample[cols].to_json("data/recipes_sample.jsonl", orient="records", lines=True)

print(len(sample), "recipes saved")

3000 recipes saved
